# Workout Recommendation ML Research Project
## Complete Jupyter Notebook

In [ ]:
!pip install pandas numpy scikit-learn xgboost shap matplotlib seaborn scipy imbalanced-learn joblib

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
    GridSearchCV
)

from sklearn.preprocessing import LabelEncoder, StandardScaler

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    roc_curve
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier
)

from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

from xgboost import XGBClassifier

from scipy.stats import ttest_rel

import joblib

In [ ]:
# LOAD DATASET

DATA_PATH = 'fitness_workout_dataset.csv'

df = pd.read_csv(DATA_PATH)

print(df.shape)

df.head()

In [ ]:
# DATASET OVERVIEW

print(df.info())

print(df.describe())

print(df.isnull().sum())

In [ ]:
# REMOVE DUPLICATES

print('Before:', len(df))

df.drop_duplicates(inplace=True)

print('After:', len(df))

In [ ]:
# REMOVE CRITICAL MISSING VALUES

critical_cols = [
    'age',
    'gender',
    'height_cm',
    'weight_kg',
    'workout_type'
]

for col in critical_cols:
    df = df[df[col].notna()]

print(df.shape)

In [ ]:
# FEATURE ENGINEERING

df['bmi'] = df['weight_kg'] / ((df['height_cm'] / 100) ** 2)

def bmi_category(bmi):

    if bmi < 18.5:
        return 'underweight'

    elif bmi < 25:
        return 'normal'

    elif bmi < 30:
        return 'overweight'

    else:
        return 'obese'

df['bmi_category'] = df['bmi'].apply(bmi_category)

df.head()

In [ ]:
# TARGET DISTRIBUTION

plt.figure(figsize=(6,4))

sns.countplot(x=df['workout_type'])

plt.title('Workout Type Distribution')
plt.show()

In [ ]:
# CORRELATION MATRIX

numeric_df = df.select_dtypes(include=['int64', 'float64'])

plt.figure(figsize=(14,10))

sns.heatmap(
    numeric_df.corr(),
    annot=True,
    cmap='coolwarm'
)

plt.title('Correlation Matrix')

plt.show()

In [ ]:
# FEATURES AND TARGET

TARGET = 'workout_type'

FEATURES = [
    'age',
    'gender',
    'height_cm',
    'weight_kg',
    'bmi',
    'fitness_level',
    'primary_goal',
    'available_equipment',
    'sessions_per_week',
    'time_per_session_min',
    'health_conditions',
    'sleep_hours',
    'stress_level',
    'bmi_category'
]

X = df[FEATURES]
y = df[TARGET]

In [ ]:
# ENCODE CATEGORICAL FEATURES

categorical_cols = X.select_dtypes(include=['object']).columns

label_encoders = {}

for col in categorical_cols:

    le = LabelEncoder()

    X[col] = le.fit_transform(X[col].astype(str))

    label_encoders[col] = le

In [ ]:
# TRAIN TEST SPLIT

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
# FEATURE SCALING

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# DEFINE MODELS

models = {

    'Logistic Regression': LogisticRegression(max_iter=1000),

    'Decision Tree': DecisionTreeClassifier(
        max_depth=8,
        random_state=42
    ),

    'Random Forest': RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        random_state=42
    ),

    'Gradient Boosting': GradientBoostingClassifier(
        random_state=42
    ),

    'AdaBoost': AdaBoostClassifier(
        random_state=42
    ),

    'SVM': SVC(
        probability=True,
        random_state=42
    ),

    'KNN': KNeighborsClassifier(
        n_neighbors=7
    ),

    'Naive Bayes': GaussianNB(),

    'XGBoost': XGBClassifier(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='logloss',
        random_state=42
    )
}

In [ ]:
# TRAIN AND EVALUATE MODELS

results = []

for name, model in models.items():

    print('=' * 60)
    print(name)
    print('=' * 60)

    if name in ['Logistic Regression', 'SVM', 'KNN', 'Naive Bayes']:

        model.fit(X_train_scaled, y_train)

        preds = model.predict(X_test_scaled)

        probs = model.predict_proba(X_test_scaled)[:,1]

    else:

        model.fit(X_train, y_train)

        preds = model.predict(X_test)

        probs = model.predict_proba(X_test)[:,1]

    accuracy = accuracy_score(y_test, preds)
    precision = precision_score(y_test, preds)
    recall = recall_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    roc_auc = roc_auc_score(y_test, probs)

    print(classification_report(y_test, preds))

    results.append({
        'Model': name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1': f1,
        'ROC_AUC': roc_auc
    })

In [ ]:
# RESULTS TABLE

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by='F1',
    ascending=False
)

results_df

In [ ]:
# CROSS VALIDATION

xgb_model = models['XGBoost']

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_scores = cross_val_score(
    xgb_model,
    X,
    y,
    cv=cv,
    scoring='f1'
)

print(cv_scores)
print('Mean F1:', cv_scores.mean())

In [ ]:
# HYPERPARAMETER OPTIMIZATION

param_grid = {
    'max_depth': [3,5,7],
    'learning_rate': [0.01,0.05,0.1],
    'n_estimators': [100,300,500]
}

grid_search = GridSearchCV(
    estimator=XGBClassifier(eval_metric='logloss'),
    param_grid=param_grid,
    scoring='f1',
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print(grid_search.best_params_)

best_xgb = grid_search.best_estimator_

In [ ]:
# FINAL EVALUATION

final_preds = best_xgb.predict(X_test)
final_probs = best_xgb.predict_proba(X_test)[:,1]

print('Accuracy:', accuracy_score(y_test, final_preds))
print('Precision:', precision_score(y_test, final_preds))
print('Recall:', recall_score(y_test, final_preds))
print('F1:', f1_score(y_test, final_preds))
print('ROC AUC:', roc_auc_score(y_test, final_probs))

In [ ]:
# CONFUSION MATRIX

cm = confusion_matrix(y_test, final_preds)

plt.figure(figsize=(6,5))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')

plt.show()

In [ ]:
# ROC CURVE

fpr, tpr, thresholds = roc_curve(y_test, final_probs)

plt.figure(figsize=(7,6))

plt.plot(fpr, tpr)

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')

plt.show()

In [ ]:
# FEATURE IMPORTANCE

importance_df = pd.DataFrame({
    'Feature': FEATURES,
    'Importance': best_xgb.feature_importances_
})

importance_df = importance_df.sort_values(
    by='Importance',
    ascending=False
)

importance_df

In [ ]:
# PLOT FEATURE IMPORTANCE

plt.figure(figsize=(10,6))

sns.barplot(
    data=importance_df,
    x='Importance',
    y='Feature'
)

plt.title('Feature Importance')

plt.show()

In [ ]:
# SHAP EXPLAINABILITY

explainer = shap.TreeExplainer(best_xgb)

shap_values = explainer.shap_values(X_test)

shap.summary_plot(shap_values, X_test)

shap.summary_plot(
    shap_values,
    X_test,
    plot_type='bar'
)

In [ ]:
# SAVE FINAL MODEL

joblib.dump(best_xgb, 'best_workout_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(label_encoders, 'label_encoders.pkl')

print('Model saved successfully.')

In [ ]:
# EXAMPLE PREDICTION

sample_user = pd.DataFrame({
    'age': [28],
    'gender': [0],
    'height_cm': [178],
    'weight_kg': [82],
    'bmi': [25.9],
    'fitness_level': [0],
    'primary_goal': [0],
    'available_equipment': [0],
    'sessions_per_week': [4],
    'time_per_session_min': [45],
    'health_conditions': [0],
    'sleep_hours': [7],
    'stress_level': [1],
    'bmi_category': [2]
})

prediction = best_xgb.predict(sample_user)

print('Prediction:', prediction[0])